# 07 · Final results on real data — stages 6.2 → 6.6 (Colab T4)

Runs every remaining compute step on a Colab GPU. Open this file in **VS Code**, pick
**Select Kernel → Colab → New Colab Server → GPU (T4)**, fill in the two links in
cell 2, then **Run All**. About 30–45 min end to end.

| Step | Produces (in `results/final/`) |
|---|---|
| 6.2 masks + Dice | `seg_val/`, `seg_calib/`, `masks/` |
| 6.3 → 6.4 oracle (ground-truth ROI) | `oracle/` |
| 6.3 → 6.4 real (predicted ROI) | `real/` |
| 6.5 Grad-CAM++ + F-Fidelity | `stage65/` |
| 6.6 demo | public Gradio link for screenshots |

Everything is also **printed and displayed in this notebook**, so the saved `.ipynb` on your
laptop keeps every number and figure even after the Colab runtime is gone.

## 1 · Runtime, code, packages

In [ ]:
import os, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

REPO = "/content/Capstone"
BRANCH = "feat/stage63-msd-negatives"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/ekkerihasan/Capstone.git {REPO}
%cd {REPO}
!git pull -q
!git log --oneline -1
# Colab already ships torch, numpy, pandas, matplotlib, scikit-learn
!pip -q install "monai[nibabel,skimage]>=1.3" pytorch-ignite SimpleITK gdown "gradio>=4.0"

## 2 · Checkpoints

`.pt` files are not in git. Upload each `best.pt` to Google Drive, **Share → Anyone with the
link**, and paste the links below.

- `STAGE62_LINK` — `checkpoints/stage62/best.pt` (Attention U-Net from the lab PC, 71 MB)
- `STAGE63_LINK` — Hasan's `checkpoints/stage63/best.pt` (classifier)

Alternative: right-click the files in the VS Code Explorer → **Upload to Colab**, and put them
at `/content/upload/stage62_best.pt` and `/content/upload/stage63_best.pt`.

In [ ]:
STAGE62_LINK = ""   # e.g. https://drive.google.com/file/d/XXXX/view?usp=sharing
STAGE63_LINK = ""

import torch
from pathlib import Path

def fetch(link, uploaded, dest):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest
    if Path(uploaded).exists():
        dest.write_bytes(Path(uploaded).read_bytes())
    elif link:
        !gdown -q --fuzzy "{link}" -O "{dest}"
    if not dest.exists():
        raise SystemExit(f"missing {dest}: paste its Drive link above or upload it to {uploaded}")
    return dest

seg_ckpt = fetch(STAGE62_LINK, "/content/upload/stage62_best.pt", "checkpoints/stage62/best.pt")
cls_ckpt = fetch(STAGE63_LINK, "/content/upload/stage63_best.pt", "checkpoints/stage63/best.pt")

seg = torch.load(seg_ckpt, map_location="cpu", weights_only=False)
cls = torch.load(cls_ckpt, map_location="cpu", weights_only=False)
print("6.2:", "epoch", seg["epoch"], "| best mean Dice", seg["best_metric"], "|", seg["extra"])
print("6.3:", "epoch", cls["epoch"], "| best", cls.get("best_metric"), "|", cls["extra"])
CLS_DROPOUT = cls["extra"].get("dropout", 0.0)
MC = "--mc 20" if CLS_DROPOUT > 0 else ""
print("MC Dropout:", "T=20" if MC else "OFF - classifier was trained without dropout")

## 3 · Data — MSD Task07 straight from the source (≈3 min on Colab)

In [ ]:
if not os.path.exists("data/Task07_Pancreas/dataset.json"):
    !mkdir -p data && wget -q -O data/Task07_Pancreas.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar
    !tar -xf data/Task07_Pancreas.tar -C data && rm data/Task07_Pancreas.tar
!ls data/Task07_Pancreas/imagesTr | grep -v '^\._' | wc -l
SPLIT = "results/review1_local/full281/split_281cases.json"

## 4 · Stage 6.2 — predicted masks and Dice (val = test group, calib = calibration group)

In [ ]:
for g in ("val", "calib"):
    !python -m src.predict_seg --split {SPLIT} --group {g} --save-masks results/final/masks --out results/final/seg_{g} --device cuda --max-figures 4

In [ ]:
import pandas as pd
from IPython.display import Image, display
print("Stage 6.2 Dice on the val group, overall and by tumor size")
display(pd.read_csv("results/final/seg_val/seg_by_size_bucket.csv"))
for f in sorted(Path("results/final/seg_val").glob("pred_*.png"))[:2]:
    display(Image(str(f)))

## 5 · Stage 6.3 → 6.4, oracle (ROIs from ground truth) and real (ROIs from the 6.2 masks)
The gap between the two is the cost of segmentation error.

In [ ]:
!python -m src.predict_cls --split {SPLIT} --ckpt {cls_ckpt} {MC} --groups val calib --out results/final/oracle --device cuda
!python -m src.run_stage64 --predictions results/final/oracle/cls_predictions.csv --out results/final/oracle
!python -m src.predict_cls --split {SPLIT} --ckpt {cls_ckpt} {MC} --groups val calib --pred-masks results/final/masks --out results/final/real --device cuda
!python -m src.run_stage64 --predictions results/final/real/cls_predictions.csv --out results/final/real

In [ ]:
import json
for name in ("oracle", "real"):
    print(f"===== {name} =====")
    print(json.dumps(json.loads(Path(f"results/final/{name}/stage64_summary.json").read_text()), indent=1)[:3000])
for f in sorted(Path("results/final/real").glob("*.png")):
    print(f.name); display(Image(str(f)))

## 6 · Stage 6.5 — Grad-CAM++ + F-Fidelity on every val patch (real masks)

In [ ]:
!python -m src.run_stage65 --split {SPLIT} --ckpt {cls_ckpt} --pred-masks results/final/masks --group val --out results/final/stage65 --device cuda
print(json.dumps(json.loads(Path("results/final/stage65/stage65_summary.json").read_text()), indent=1))
for f in sorted(Path("results/final/stage65").glob("fig_saliency_*.png")):
    display(Image(str(f)))

## 7 · Stage 6.6 — demo with the real weights
Prints a public `*.gradio.live` link. Open it in your browser and take screenshots of three
cases: a detected sub-2 cm tumor, a large tumor, and one **flagged for specialist review**.
The banner must list both checkpoints and say **Real CT**.

In [ ]:
from src.app import build_ui
ui = build_ui(Path("data/Task07_Pancreas"), checkpoint=str(seg_ckpt), classifier=str(cls_ckpt))
ui.launch(share=True, prevent_thread_lock=True)

## 8 · Bring the results home
Everything except the 84 mask arrays, zipped. In VS Code's **Colab** file view, right-click
`/content/Capstone/final_results.zip` → **Download**. If that fails, the tables and figures above
are already saved in this notebook.

In [ ]:
!cd results && zip -qr ../final_results.zip final -x "final/masks/*"
!ls -lh final_results.zip
!find results/final -maxdepth 2 -not -path "*/masks/*" | sort